# 143 — Context Compression

**What this workbook teaches:**
- Why context compression matters for long documents and limited context windows
- How `split_sentences()` and `count_tokens()` work as the preprocessing layer
- How the LLM-scored compression pipeline filters sentences by relevance to a query
- How to quantify the before/after token reduction

---

**Two sections:**
- **Part A** — Pure Python, no API key required. Explore the pipeline with mock scores.
- **Part B** — Live OpenAI calls. Requires `OPENAI_API_KEY` in your environment.

In [ ]:
%pip install -q openai python-dotenv langgraph

---
## Part A — Concepts (no API key needed)

### Why context compression?

LLMs have finite context windows. When answering a question about a long document, most of the text is *not relevant* to the specific question asked. Sending the full document wastes tokens (and money), increases latency, and can dilute the model's focus.

**Context compression** addresses this by:
1. **Splitting** the document into sentences
2. **Scoring** each sentence for relevance to the query (using an LLM judge)
3. **Keeping** only the top N% of sentences (controlled by `ratio`)
4. **Sending** the compressed context to the answering LLM

The key insight is that a **small, focused context** often produces *better* answers than a large, noisy one — and at a fraction of the token cost.

**The sentence scoring approach:**
- Ask the LLM: "Rate each sentence 0.0–1.0 for relevance to this query."
- Sort sentences by score, keep the top `ratio` fraction
- Default `ratio = 0.4` means keep 40% of sentences

**Trade-off:** The compression step itself uses LLM tokens. Worthwhile when:
- The document is long (many sentences)
- The same compressed context is reused for multiple queries
- The answering model is more expensive than the scoring model

In [ ]:
# Part A — Cell 2: Demo split_sentences() and count_tokens()
# These are the exact functions from src/tools.py — no LLM needed.

import re

def split_sentences(text: str) -> list[str]:
    """Split text into sentences on sentence-ending punctuation."""
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def count_tokens(text: str) -> int:
    """Rough token estimate: ~4 chars per token."""
    return len(text) // 4


SAMPLE_PARAGRAPH = (
    "The Python programming language was created by Guido van Rossum and first released in 1991. "
    "Python's design philosophy emphasizes code readability with the use of significant indentation. "
    "It supports multiple programming paradigms, including structured, object-oriented, and functional programming. "
    "CPython is the reference implementation of Python. "
    "The GIL in CPython prevents true parallel execution of Python threads. "
    "Python is dynamically typed and garbage-collected. "
    "It features a comprehensive standard library often described as batteries included."
)

sentences = split_sentences(SAMPLE_PARAGRAPH)

print(f"Input: {len(SAMPLE_PARAGRAPH)} chars, ~{count_tokens(SAMPLE_PARAGRAPH)} tokens")
print(f"Sentences found: {len(sentences)}")
print()
for i, s in enumerate(sentences):
    print(f"  [{i+1}] ({count_tokens(s):>3} tok) {s}")

In [ ]:
# Part A — Cell 3: Compression pipeline with MOCK scores (no LLM call)
# Shows exactly which sentences survive at ratio=0.4

QUERY = "How does CPython execute Python code?"

# Hardcoded relevance scores (0.0 = irrelevant, 1.0 = highly relevant)
# Pretending an LLM rated each sentence against the query above.
MOCK_SCORES = [
    0.1,   # creation by Guido — not about execution
    0.05,  # design philosophy — not about execution
    0.15,  # programming paradigms — tangentially relevant
    0.90,  # CPython is reference implementation — very relevant
    0.85,  # GIL prevents parallel threads — relevant to CPython execution
    0.20,  # dynamically typed — somewhat relevant
    0.05,  # standard library — not about execution
]

assert len(MOCK_SCORES) == len(sentences), "Score count must match sentence count"

scored = list(zip(MOCK_SCORES, sentences))
scored_sorted = sorted(scored, key=lambda x: x[0], reverse=True)

ratio = 0.4
keep_n = max(1, int(len(scored_sorted) * ratio))
kept = [s for _, s in scored_sorted[:keep_n]]
dropped = [s for _, s in scored_sorted[keep_n:]]

compressed_text = " ".join(kept)

print(f"Query: {QUERY}")
print(f"Ratio: {ratio}  |  Keep top {keep_n} of {len(scored)} sentences")
print()
print("=== Scores (sorted) ===")
for score, s in scored_sorted:
    tag = "KEEP" if (score, s) in [(sc, se) for sc, se in scored_sorted[:keep_n]] else "DROP"
    print(f"  [{tag}] {score:.2f}  {s[:70]}")

print()
print("=== Compressed context ===")
print(compressed_text)
print()
print(f"Original:   ~{count_tokens(SAMPLE_PARAGRAPH)} tokens")
print(f"Compressed: ~{count_tokens(compressed_text)} tokens")
reduction = round((1 - count_tokens(compressed_text) / count_tokens(SAMPLE_PARAGRAPH)) * 100)
print(f"Reduction:  {reduction}%")

### Token count before/after — sample comparison

Here's what the compression output looks like across different ratios on the same document:

| Ratio | Sentences kept | Approx. tokens | Reduction |
|---|---|---|---|
| 1.0 (no compression) | 7 / 7 | ~87 | 0% |
| 0.6 | 4 / 7 | ~54 | ~38% |
| 0.4 (default) | 3 / 7 | ~40 | ~54% |
| 0.2 | 2 / 7 | ~26 | ~70% |

For a **500-word document** (~625 tokens), a 0.4 ratio compresses to ~250 tokens — saving ~375 tokens per LLM call.

In [ ]:
# Part A — Cell 4 bonus: Sweep ratios to verify the table above

original_tokens = count_tokens(SAMPLE_PARAGRAPH)

print(f"{'Ratio':<8} {'Kept':>6} {'Tokens':>8} {'Reduction':>10}")
print("-" * 38)
for ratio in [1.0, 0.6, 0.4, 0.2]:
    keep_n = max(1, int(len(scored_sorted) * ratio))
    kept_sentences = [s for _, s in scored_sorted[:keep_n]]
    compressed = " ".join(kept_sentences)
    tok = count_tokens(compressed)
    pct = round((1 - tok / original_tokens) * 100)
    print(f"{ratio:<8} {keep_n:>6}/{len(sentences)} {tok:>8} {pct:>9}%")

---
## Part B — Live OpenAI calls

**Requires:** `OPENAI_API_KEY` in your `.env` file or environment.

Runs the full compression + answer workflow on a ~500-word Python document:
1. LLM scores each sentence for relevance to the query
2. Top 40% of sentences are kept
3. Compressed context is sent to the answering LLM
4. Prints original vs. compressed token counts and the final answer

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY is not set. "
        "Add it to your .env file or export it before running Part B."
    )
print("OPENAI_API_KEY loaded.")

In [ ]:
# Part B — Run full compress + answer workflow on a ~500-word document
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "."))

from openai import OpenAI
from src.workflow import create_workflow

SAMPLE_CHUNKS = [
    "The Python programming language was created by Guido van Rossum and first released in 1991. "
    "Python's design philosophy emphasizes code readability with the use of significant indentation. "
    "It supports multiple programming paradigms, including structured, object-oriented, and functional programming.",

    "Python is dynamically typed and garbage-collected. It features a comprehensive standard library. "
    "The language is often described as batteries included due to this comprehensive library. "
    "Python consistently ranks as one of the most popular programming languages worldwide.",

    "CPython is the reference implementation of Python. When people refer to Python they often mean CPython. "
    "Other implementations include PyPy, Jython, and IronPython. "
    "CPython is an interpreter that compiles Python to bytecode before executing it. "
    "The GIL in CPython prevents true parallel execution of Python threads.",

    "Python 3 was released in 2008 and is incompatible with Python 2 in several ways. "
    "Python 2 reached end-of-life on January 1, 2020. "
    "The transition from Python 2 to Python 3 took the community over a decade to complete. "
    "Most major libraries have since dropped Python 2 support.",

    "Python's async/await syntax was introduced in Python 3.5 via PEP 492. "
    "asyncio is the standard library module for writing concurrent code with async/await. "
    "It uses an event loop to manage I/O-bound tasks without threading. "
    "This makes Python suitable for building high-performance network servers and web applications.",
]

QUERY = "How does CPython execute Python code?"

client = OpenAI(api_key=OPENAI_API_KEY)
workflow = create_workflow()
config = {"configurable": {"client": client, "model": "gpt-4o-mini"}}

result = workflow.invoke({
    "chunks": SAMPLE_CHUNKS,
    "query": QUERY,
    "ratio": 0.4,
    "original_tokens": 0,
    "compressed_context": "",
    "compressed_tokens": 0,
    "answer": "",
}, config=config)

orig = result["original_tokens"]
comp = result["compressed_tokens"]
reduction = round((1 - comp / orig) * 100) if orig else 0

print(f"Query: {QUERY}")
print()
print(f"{'Metric':<25} {'Value'}")
print("-" * 45)
print(f"{'Original tokens':<25} {orig}")
print(f"{'Compressed tokens':<25} {comp}")
print(f"{'Token reduction':<25} {reduction}%")
print(f"{'Tokens saved':<25} {orig - comp}")
print()
print("=== Answer (from compressed context) ===")
print(result["answer"])